# First historical development results

This notebook reads the integrated Experiment 005 results and the separately registered
Experiment 006 spot-benchmark correction. It does not train models or execute trades.

- **Momentum baseline:** 2021–2022, no fitting.
- **LightGBM validation:** trained once on purged 2021 labels, evaluated in 2022.
- **Primary cost case:** 5 bps adverse slippage per fill; all 0/2/5/10 bps cases are retained.
- **Benchmarks:** genuine BTC/ETH spot prices, fixed-unit passive hold, 0.1% reference fees,
  5 bps adverse entry/exit fills, no leverage or funding. These are passive accounting
  benchmarks, not Nautilus spot-exchange simulations.
- **Scope:** conditional 37-symbol pre-2021 cohort. Identity continuity, fractional lots,
  hourly VWAP execution and funding settlement marks are research assumptions.
  This is **not a certified point-in-time universe, deployable strategy, or final OOS result**.

Do not compare the two strategy periods as though they were the same evaluation window.
2023, 2024 and the final OOS remain unopened.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/momentum_data/backtest_analysis.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the project repository")
sys.path.insert(0, str(ROOT / "src"))
from momentum_data.backtest_analysis import load_development_run, plot_development  # noqa: E402

RUN = ROOT / "data/interim/first_development/v20261121_001/results_v003"
FIGURES = ROOT / "reports/figures/first_development"
TABLES = ROOT / "reports/tables/first_development"
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)
if not (RUN / "summary.json").is_file():
    raise RuntimeError("The integrated run is incomplete. See docs/FIRST_HISTORICAL_DEVELOPMENT.md.")
summary = json.loads((RUN / "summary.json").read_text())
assert summary["success"] and summary["historical_backtest_executed"]
assert summary["OOS_performance_inspected"] is False
display(pd.Series({k: summary[k] for k in [
    "registered_runs_completed", "primary_replay_identical",
    "certified_point_in_time_universe", "paper_or_live_ready", "OOS_performance_inspected",
]}, name="Run status"))
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .2})

## 1. All registered runs

Rows are sorted by strategy and cost assumption, not by performance.
The primary case was fixed before PnL inspection. Losses and weaker cases remain visible.
Returns are net of strategy commissions and modeled funding. Costs are current-rate
research assumptions, not authenticated historical account fees.

In [ ]:
all_runs = pd.read_csv(RUN / "all_runs.csv").sort_values(["model", "slippage_bps"])
columns = ["model", "start_year", "slippage_bps", "total_return", "CAGR",
           "Sharpe_rf0", "MDD", "Calmar", "fill_count", "funding_cashflows", "ledger_error_usdt"]
display(all_runs[columns])
all_runs.to_csv(TABLES / "all_registered_runs.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for name, group in all_runs.groupby("model"):
    axes[0].plot(group.slippage_bps, group.total_return, marker="o", label=name)
    axes[1].plot(group.slippage_bps, group.MDD, marker="o", label=name)
axes[0].set(title="Net return sensitivity", xlabel="Adverse slippage per fill (bps)", ylabel="Return")
axes[1].set(title="Drawdown sensitivity", xlabel="Adverse slippage per fill (bps)", ylabel="MDD")
for ax in axes:
    ax.legend()
fig.suptitle("Different strategy periods: compare costs within each strategy")
fig.savefig(FIGURES / "cost_sensitivity.png", dpi=160)
plt.show()

## 2. Primary-case dashboards

Each dashboard compares the strategy with BTC and ETH spot-price hold on its own date grid.
Daily NAV includes unrealized PnL, commissions and modeled funding. The final position is
closed on 2022-12-30 at 23:00 UTC; the final valuation is 2022-12-31 at 00:00 UTC.
Funding is applied at recorded event times using the latest completed mark-hour close;
this is modeled settlement, not proof of paid historical cashflow.

Annualization uses 365 days and a zero risk-free rate. Missing dates or invalid NAV raise
errors instead of being forward-filled. Undefined ratios are NaN.

In [ ]:
primary_runs = {
    "Momentum baseline: 2021–2022": RUN / "momentum_baseline/slippage_5bps",
    "LightGBM validation: 2022": RUN / "lightgbm_validation/slippage_5bps",
}
for title, directory in primary_runs.items():
    daily, manifest = load_development_run(directory)
    print(title)
    print(manifest["benchmark_definition"])
    fig, metrics, monthly = plot_development(daily)
    fig.suptitle(title + " — conditional research, BTC/ETH spot benchmarks")
    display(metrics)
    costs = pd.Series({
        "commissions_USDT": daily.commissions_usdt.sum(),
        "funding_cashflow_USDT": daily.funding_cashflow_usdt.sum(),
        "total_traded_notional_USDT": daily.traded_notional_usdt.sum(),
    }, name="Accounting totals")
    display(costs)
    stem = directory.parent.name
    fig.savefig(FIGURES / (stem + "_primary_dashboard.png"), dpi=160)
    metrics.to_csv(TABLES / (stem + "_metrics.csv"))
    monthly.to_csv(TABLES / (stem + "_monthly_returns.csv"))
    costs.to_csv(TABLES / (stem + "_costs.csv"))
    plt.show()

## 3. Model provenance and SHAP

Training labels must be known before 2022-01-01. The existing fitted model was reused
after the benchmark-only correction; no performance-based refitting or tuning is allowed.
SHAP explains the model's predicted return, **not causal factor effects or portfolio weights**.
The three momentum SHAP values are summed per row before computing grouped importance.

In [ ]:
model = json.loads((RUN / "model/model_summary.json").read_text())
display(pd.Series({k: model[k] for k in [
    "train_rows", "validation_rows", "latest_training_label_available_ms",
    "validation_start_ms", "fit_count", "shap_additivity_max_error",
]}, name="Model provenance"))
display(pd.Series(model["parameters"], name="Frozen LightGBM parameters"))
attribution = pd.read_csv(RUN / "model/shap_values.csv")
shap_columns = [c for c in attribution if c.startswith("shap_")]
residual = (attribution[shap_columns].sum(axis=1)
            + attribution.expected_value - attribution.score).abs().max()
assert residual < 1e-6
feature_importance = attribution[shap_columns].abs().mean().sort_values()
grouped = pd.DataFrame({
    "Momentum": attribution[["shap_momentum20", "shap_momentum60", "shap_momentum120"]].sum(axis=1),
    "Funding": attribution.shap_funding7,
    "Aggressive imbalance": attribution.shap_imbalance7,
})
group_importance = grouped.abs().mean().sort_values()
display(group_importance.rename("Mean absolute grouped SHAP"))
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
feature_importance.plot.barh(ax=axes[0], color="#2a9d8f")
group_importance.plot.barh(ax=axes[1], color="#457b9d")
axes[0].set(title="Five model inputs", xlabel="Mean absolute SHAP")
axes[1].set(title="Three factor components", xlabel="Mean absolute grouped SHAP")
fig.suptitle("2022 validation attribution — not causal importance")
fig.savefig(FIGURES / "validation_shap.png", dpi=160)
group_importance.to_csv(TABLES / "grouped_shap_importance.csv")
plt.show()

## 4. Trade and funding ledgers

Inspect individual fills, fees, funding events, sizing references and portfolio targets.
Source hashes and registered assumptions are retained alongside the run.
The initial failed funded-perpetual benchmark remains in the original results/diagnostics
directories; it is not substituted for spot buy-and-hold.

In [ ]:
primary = RUN / "lightgbm_validation/slippage_5bps"
for name in ("fills", "funding", "targets"):
    frame = pd.read_csv(primary / (name + ".csv"))
    print(name, "rows:", len(frame))
    display(frame.head(10))
    frame.to_csv(TABLES / ("lightgbm_" + name + ".csv"), index=False)
print("Figures:", FIGURES)
print("Tables:", TABLES)
print("All outputs are development research artifacts. OOS remains sealed.")